# 42 · Are the S1/S2/S3 labels anatomical in both species?

Referee issue M1: the reviewed S1/S2/S3 labels are joint Leiden clusters on a Harmony embedding in
which specimen, and therefore species, is the batch. A human structure is "S2" because Harmony
co-clusters it with mouse S2. Human S1 is 55% of human PT against 33% in mouse. This notebook asks,
without the cross-species embedding, whether the labels agree with anatomy and with external
segment references, and it measures two data-quality issues (D7):

| Part | Question |
|---|---|
| Geometry | Can segmentation polygons be joined to our structures, and are the glomeruli glomerulus-sized? |
| (a) | Is S1 closest to glomeruli, in both species? |
| (b) | Are PT polygons touching a glomerulus (S1 at the urinary pole) labelled S1 in both species? |
| (c) | Which S1/S2/S3 does expression alone assign each structure to, against Lake/KPMP (human) and microdissection/snRNA (mouse)? |
| (d) | Alternative per-structure labels saved for workstream 2 |
| Classes | How many integration genes fall in each notebook-40 zonation class? |
| D7 | Non-PT marker spillover in PT structures; human polygons without bins |

**Protocol.** Fixed in the workstream-1 plan before any output (revision-round protocol and its
glomerulus-source amendment). Everything is descriptive. Two control mice and two cortex sections
from one human donor.

## 1 · Folders and fixed settings

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
output = results_root / 'pt_revision_labels'
figures = output / 'figures'
figures.mkdir(parents=True, exist_ok=True)
SAMPLES = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
SEGMENTATION = {'Ctrl1A2': 'Ctrl1A2_kept_tubules_labeled_fine.geojson', 'Ctrl1A4': 'Ctrl1A4_kept_tubules_labeled_fine.geojson',
                'HUK1_COR1': 'HUK1_COR1_v2.geojson', 'HUK1_MED1': 'HUK1_MED1_v2.geojson'}
GLOMERULUS_GEOMETRY = {'Ctrl1A2': 'Ctrl_1A2_v4.geojson', 'Ctrl1A4': 'Ctrl_1A4_v4.geojson',   # geometry only
                       'HUK1_COR1': 'HUK1_COR1_v2.geojson', 'HUK1_MED1': 'HUK1_MED1_v2.geojson'}
source_paths = {n: data_root / 'tubule_by_gene' / f'{n}_tubule_by_gene_caleb.h5ad' for v in SAMPLES.values() for n in v}
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            results_root / 'pt_literature_deep' / 'lake_cortex_segment_means.csv',
            results_root / 'pt_zonation_classes' / 'tables' / 'gene_zonation_classes.csv',
            data_root / 'external' / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            data_root / 'external' / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            *source_paths.values(), *[data_root / f for f in {**SEGMENTATION, **GLOMERULUS_GEOMETRY}.values()]]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '42.label_validation.1'
CENTROID_TOLERANCE_PX = 2.0
ATTACH_UM, ATTACH_UM_SENSITIVITY = 2.0, 10.0
MIN_DETECTION, MIN_REFERENCE_LEVEL, MIN_REFERENCE_DIFFERENCE = .10, 1.0, 1.0
MARKERS = {'AL': ['Umod', 'Slc12a1'], 'DCT': ['Slc12a3', 'Pvalb'], 'CNT/CD': ['Aqp2', 'Calb1', 'Scnn1g'],
           'podocyte': ['Nphs1', 'Nphs2', 'Podxl'], 'endothelium': ['Pecam1', 'Emcn', 'Kdr'], 'immune': ['Ptprc'],
           'stroma': ['Pdgfrb', 'Col1a1', 'Col3a1'], 'smooth muscle': ['Acta2', 'Myh11']}
MOUSE_BIN_UM = 2.0   # documented 2 um square bins for the mouse Visium HD outputs
HUMAN, MOUSE, INK = '#D55E00', '#0072B2', '#333333'
print('Results folder:', output)

## 2 · Structures, labels and polygons

Notebook 13's pass-1 cohort gives every structure's specimen, centroid, `feature_index` and
reviewed label (PT S1/S2/S3, Glomerulus and the other classes). `feature_index` is a positional
index into the GeoJSON that notebook 01 enumerated. The join is accepted only if every polygon
centroid lies within 2 px of the stored centroid. Pixel size comes from each feature's own area in
pixels and µm².

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from scipy.stats import rankdata, spearmanr
from shapely import STRtree
from shapely.geometry import shape
from sklearn.metrics import cohen_kappa_score

from pseudospace.anchor_coordinate import bulk_segment_profiles, census_segment_profiles
from pseudospace.pathway_inputs import rebuild_pt_expression

REVIEWED_13 = {'0': 'PT-S1', '1': 'PT-S2', '2': 'CNT_CD', '3': 'AL', '4': 'DCT', '5': 'Glomerulus', '6': 'Unassigned',
               '7': 'PT-S3', '8': 'CCD', '9': 'AL', '10': 'SmoothMuscle'}  # notebook 13's reviewed map
key = [p for p in sorted((upstream13 / 'stage_cache').glob('reviewed_embedding__*.h5ad.key.json'))
       if "'logic': '13-nephron-scfates-v2'" in json.loads(p.read_text())['params']]
if len(key) != 1:
    raise FileNotFoundError('Expected one notebook 13 reviewed-embedding cache; rerun notebook 13.')
cohort = ad.read_h5ad(str(key[0])[:-len('.key.json')], backed='r').obs[
    ['sample', 'feature_index', 'x_centroid', 'y_centroid', 'n_spots', 'total_counts', 'comparison_species', 'leiden_coarse']].copy()
cohort['label'] = cohort.leiden_coarse.astype(str).map(REVIEWED_13)
saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r').obs.copy()
if not saved_pt.index.isin(cohort.index).all() or not (cohort.loc[saved_pt.index, 'label'] == saved_pt.segment_class.astype(str)).all():
    raise ValueError('Cohort labels do not reproduce notebook 13 PT labels.')
cohort['sample'] = cohort['sample'].astype(str)

geometry, join_rows, um_per_px = {}, [], {}
for name, file in SEGMENTATION.items():
    features = json.loads((data_root / file).read_text())['features']
    polys = [shape(f['geometry']) for f in features]
    ratio = [np.sqrt(next(m['value'] for m in f['properties']['measurements'] if m['name'] == 'Area um^2')
                     / next(m['value'] for m in f['properties']['measurements'] if m['name'] == 'Area px')) for f in features]
    um_per_px[name] = float(np.median(ratio))
    rows = cohort[cohort['sample'].eq(name)]
    index = rows.feature_index.to_numpy(int)
    if index.max() >= len(polys):
        raise ValueError(f'{name}: feature_index exceeds the segmentation.')
    centroids = np.array([polys[i].centroid.coords[0] for i in index])
    offset = np.hypot(*(centroids - rows[['x_centroid', 'y_centroid']].to_numpy(float)).T)
    join_rows.append({'specimen': name, 'structures': len(rows), 'polygons': len(polys), 'um per px': um_per_px[name],
                      'max centroid offset px': float(offset.max()), 'join safe': bool(offset.max() <= CENTROID_TOLERANCE_PX)})
    geometry[name] = pd.Series([polys[i] for i in index], index=rows.index)
JOIN = pd.DataFrame(join_rows)
JOIN.to_csv(output / 'geometry_join_check.csv', index=False)
display(JOIN)
if not JOIN['join safe'].all():
    raise ValueError('Polygon join failed the centroid check; geometry-based checks would be unsafe.')

# Morphological glomeruli (H&E segmentation class). Mouse v4 is used for geometry only; its frame
# must match the fine segmentation (centroid of each fine polygon has a v4 polygon within 2 px).
glomeruli, glom_rows = {}, []
for name, file in GLOMERULUS_GEOMETRY.items():
    features = json.loads((data_root / file).read_text())['features']
    polys = [shape(f['geometry']) for f in features if f['properties'].get('classification', {}).get('name') == 'glomerulus']
    if file != SEGMENTATION[name]:
        frame = np.array([p.centroid.coords[0] for p in (shape(f['geometry']) for f in features)])
        from scipy.spatial import cKDTree
        fine = np.array([g.centroid.coords[0] for g in geometry[name].iloc[:2000]])
        frame_offset = float(np.median(cKDTree(frame).query(fine)[0]))
    else:
        frame_offset = 0.0
    glomeruli[name] = polys
    pt_area = np.median([geometry[name][i].area for i in cohort.index[cohort['sample'].eq(name) & cohort.label.str.startswith('PT-')]])
    glom_rows.append({'specimen': name, 'morphological glomeruli': len(polys),
                      'median glomerulus area um2': float(np.median([p.area for p in polys]) * um_per_px[name] ** 2),
                      'median PT area um2': float(pt_area * um_per_px[name] ** 2),
                      'area ratio': float(np.median([p.area for p in polys]) / pt_area),
                      'expression Glomerulus structures': int((cohort['sample'].eq(name) & cohort.label.eq('Glomerulus')).sum()),
                      'frame offset px (median)': frame_offset})
GLOM = pd.DataFrame(glom_rows)
GLOM['glomerulus-sized (ratio ≥ 3)'] = GLOM['area ratio'] >= 3
GLOM.to_csv(output / 'glomerulus_polygons.csv', index=False)
display(GLOM.round(2))

## 3 · (a) Distance to the nearest glomerulus and (b) glomerulus-attached PT

Distances are polygon boundary to polygon boundary, in µm. A PT polygon within 2 µm of a
glomerulus polygon is "attached" (10 µm as sensitivity). The expression `Glomerulus` cluster is a
sensitivity source. If labels are anatomical, S1 should be nearest to glomeruli and enriched among
attached PT in both species.

In [ ]:
pt = cohort.loc[saved_pt.index].copy()
pt['species'] = pt.comparison_species.astype(str)
pt['segment'] = pt.label.str.replace('PT-', '', regex=False)
for source in ('morphological', 'expression'):
    distance = np.full(len(pt), np.nan)
    for name in pt['sample'].unique():
        rows = np.flatnonzero(pt['sample'].eq(name).to_numpy())
        if source == 'morphological':
            gpolys = glomeruli[name]
        else:
            gpolys = list(geometry[name][cohort.index[cohort['sample'].eq(name) & cohort.label.eq('Glomerulus')]])
        tree = STRtree(gpolys)
        polys = list(geometry[name].loc[pt.index[rows]])
        _, d = tree.query_nearest(polys, return_distance=True, all_matches=False)
        distance[rows] = d * um_per_px[name]
    pt[f'glomerulus_distance_um_{source}'] = distance
DIST_ROWS = []
for source in ('morphological', 'expression'):
    col = f'glomerulus_distance_um_{source}'
    for (sp, seg), block in pt.groupby(['species', 'segment']):
        DIST_ROWS.append({'glomeruli': source, 'species': sp, 'segment': seg, 'n': len(block),
                          'median um': float(block[col].median()), 'q25': float(block[col].quantile(.25)),
                          'q75': float(block[col].quantile(.75))})
    for sp in SAMPLES:
        block = pt[pt.species.eq(sp)]
        DIST_ROWS.append({'glomeruli': source, 'species': sp, 'segment': 'Spearman(label order, distance)',
                          'n': len(block), 'median um': spearmanr(block.segment.map({'S1': 0, 'S2': 1, 'S3': 2}), block[col]).statistic})
DISTANCE = pd.DataFrame(DIST_ROWS)
DISTANCE.to_csv(output / 'glomerulus_distance_by_label.csv', index=False)
display(DISTANCE.round(3))

ATTACH_ROWS = []
for source in ('morphological', 'expression'):
    col = f'glomerulus_distance_um_{source}'
    for threshold in (ATTACH_UM, ATTACH_UM_SENSITIVITY):
        for group, block in [(sp, pt[pt.species.eq(sp)]) for sp in SAMPLES] + [(n, pt[pt['sample'].eq(n)]) for n in sum(SAMPLES.values(), [])]:
            attached = block[col] <= threshold
            s1 = block.segment.eq('S1')
            a, b_, c, d = (attached & s1).sum(), (attached & ~s1).sum(), (~attached & s1).sum(), (~attached & ~s1).sum()
            ATTACH_ROWS.append({'glomeruli': source, 'threshold um': threshold, 'group': group, 'attached PT': int(attached.sum()),
                                'S1 among attached': float(a / max(attached.sum(), 1)), 'S1 among all PT': float(s1.mean()),
                                'S2 among attached': float((attached & block.segment.eq('S2')).sum() / max(attached.sum(), 1)),
                                'S3 among attached': float((attached & block.segment.eq('S3')).sum() / max(attached.sum(), 1)),
                                'odds ratio S1': float((a + .5) * (d + .5) / ((b_ + .5) * (c + .5)))})
ATTACHED = pd.DataFrame(ATTACH_ROWS)
ATTACHED.to_csv(output / 'glomerulus_attached_pt.csv', index=False)
display(ATTACHED[ATTACHED.glomeruli.eq('morphological')].round(3))

## 4 · (c) Expression-only reference transfer, species by species

No embedding and no other species are involved. Genes must be measured in both species, detected
in ≥ 10% of the species' PT structures, expressed (max log2 ≥ 1) in the reference and differ by
≥ 1 log2 between some pair of reference segments. The primary score is the Spearman correlation of
each structure's log-normalised profile with each reference segment profile (argmax). A
sensitivity score correlates the structure's profile centred on its specimen's PT mean with the
reference centred over its three segments; that centring depends on segment composition.

In [ ]:
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
expr = rebuild_pt_expression(saved_pt, source_paths, orthologs, target_sum=1e4)
X = sparse.csr_matrix(expr.layers['lognorm'], dtype=float)
genes_all = pd.Index(expr.var_names)
measured_both = expr.var.measured_in_both_inputs.to_numpy(bool)
species_pt = saved_pt.comparison_species.astype(str).to_numpy()
specimen_pt = saved_pt['sample'].astype(str).to_numpy()
detected = {sp: np.asarray((X[species_pt == sp] > 0).mean(axis=0)).ravel() for sp in SAMPLES}

lake = pd.read_csv(results_root / 'pt_literature_deep' / 'lake_cortex_segment_means.csv', index_col='gene')
lake = lake.rename(columns=lambda c: c.replace('lake_log2cpm_', '')).reindex(genes_all)
micro_table = pd.read_csv(data_root / 'external' / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
micro = bulk_segment_profiles(micro_table.drop_duplicates('Gene_symbol').set_index('Gene_symbol'), '_PT{seg}_').reindex(genes_all)[['S1', 'S2', 'S3']]
census_counts = pd.read_csv(data_root / 'external' / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
census_meta = pd.read_csv(data_root / 'external' / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_meta.csv', index_col=0)
sn = census_segment_profiles(census_counts, census_meta.loc[census_counts.index],
                             {'epithelial cell of proximal tubule segment 1': 'S1', 'epithelial cell of proximal tubule segment 2': 'S2',
                              'epithelial cell of proximal tubule segment 3': 'S3'}).reindex(genes_all)
sn = sn.loc[:, sn.columns.get_level_values('sex') == 'male']
snrna = pd.DataFrame({s: sn.xs(s, axis=1, level='segment').mean(axis=1) for s in ('S1', 'S2', 'S3')})
REFERENCES = {'human': {'Lake/KPMP cortex snRNA': lake}, 'mouse': {'microdissection (primary)': micro, 'male snRNA (Census)': snrna}}


def reference_genes(reference, sp):
    diffs = pd.concat([(reference[a] - reference[b]).abs() for a, b in (('S2', 'S1'), ('S3', 'S2'), ('S3', 'S1'))], axis=1).max(axis=1)
    keep = (measured_both & (detected[sp] >= MIN_DETECTION) & reference.notna().all(axis=1).to_numpy()
            & (reference.max(axis=1) >= MIN_REFERENCE_LEVEL).to_numpy() & (diffs >= MIN_REFERENCE_DIFFERENCE).to_numpy())
    return np.flatnonzero(keep)


def transfer(rows, reference, cols):
    """Primary (Spearman, uncentred) and sensitivity (centred Pearson) segment assignments."""
    dense = X[rows][:, cols].toarray()
    ref = reference.iloc[cols][['S1', 'S2', 'S3']].to_numpy(float)
    ranks = rankdata(dense, axis=1)
    ranks = (ranks - ranks.mean(axis=1, keepdims=True))
    ranks /= np.linalg.norm(ranks, axis=1, keepdims=True).clip(1e-12)
    ref_ranks = rankdata(ref, axis=0)
    ref_ranks = (ref_ranks - ref_ranks.mean(axis=0)) / np.linalg.norm(ref_ranks - ref_ranks.mean(axis=0), axis=0)
    primary = ranks @ ref_ranks
    centred = dense.copy()
    for n in np.unique(specimen_pt[rows]):
        m = specimen_pt[rows] == n
        centred[m] -= centred[m].mean(axis=0)
    centred /= np.linalg.norm(centred, axis=1, keepdims=True).clip(1e-12)
    ref_c = ref - ref.mean(axis=1, keepdims=True)
    ref_c /= np.linalg.norm(ref_c, axis=0, keepdims=True).clip(1e-12)
    sensitivity = centred @ ref_c
    order = np.sort(primary, axis=1)
    return (np.array(['S1', 'S2', 'S3'])[primary.argmax(axis=1)], order[:, -1] - order[:, -2],
            np.array(['S1', 'S2', 'S3'])[sensitivity.argmax(axis=1)])


current = saved_pt.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
labels_table = pd.DataFrame({'structure_id': saved_pt.index, 'specimen': specimen_pt, 'species': species_pt,
                             'current_label': current})
transfer_rows, fraction_rows = [], []
for sp, refs in REFERENCES.items():
    rows = np.flatnonzero(species_pt == sp)
    for ref_name, reference in refs.items():
        cols = reference_genes(reference, sp)
        primary, margin, centred = transfer(rows, reference, cols)
        tag = ref_name.split(' ')[0].split('/')[0].lower()
        labels_table.loc[rows, f'transfer_{tag}'] = primary
        labels_table.loc[rows, f'transfer_{tag}_margin'] = margin
        labels_table.loc[rows, f'transfer_{tag}_centred'] = centred
        for score, assigned in (('primary (Spearman)', primary), ('sensitivity (centred)', centred)):
            transfer_rows.append({'species': sp, 'reference': ref_name, 'score': score, 'genes': len(cols),
                                  'agreement with current': float((assigned == current[rows]).mean()),
                                  'Cohen kappa': float(cohen_kappa_score(current[rows], assigned))})
            confusion = pd.crosstab(pd.Series(current[rows], name='current'), pd.Series(assigned, name='transfer'))
            confusion.to_csv(output / f'transfer_confusion_{sp}_{tag}_{score.split(" ")[0]}.csv')
            for n in SAMPLES[sp]:
                m = specimen_pt[rows] == n
                fraction_rows.append({'species': sp, 'specimen': n, 'reference': ref_name, 'score': score,
                                      **{f'{s} fraction (transfer)': float((assigned[m] == s).mean()) for s in ('S1', 'S2', 'S3')},
                                      **{f'{s} fraction (current)': float((current[rows][m] == s).mean()) for s in ('S1', 'S2', 'S3')}})
TRANSFER = pd.DataFrame(transfer_rows)
FRACTIONS = pd.DataFrame(fraction_rows)
TRANSFER.to_csv(output / 'transfer_agreement.csv', index=False)
FRACTIONS.to_csv(output / 'segment_fractions.csv', index=False)
display(TRANSFER.round(3))
display(FRACTIONS.round(3))
for sp, refs in REFERENCES.items():
    tag = list(refs)[0].split(' ')[0].split('/')[0].lower()
    display(pd.read_csv(output / f'transfer_confusion_{sp}_{tag}_primary.csv', index_col=0).rename_axis(f'{sp}: current ↓ / transfer →'))
human_primary = TRANSFER[(TRANSFER.species == 'human') & TRANSFER.score.str.startswith('primary')]['agreement with current'].iloc[0]
mouse_primary = TRANSFER[(TRANSFER.species == 'mouse') & TRANSFER.score.str.startswith('primary')
                         & TRANSFER.reference.str.startswith('micro')]['agreement with current'].iloc[0]
SUPPORT = bool(human_primary >= .70 and abs(human_primary - mouse_primary) <= .10)
print(f'Pre-specified reading: human agreement {human_primary:.3f}, mouse {mouse_primary:.3f} -> '
      f'{"supports" if SUPPORT else "weak support for"} the current human labels')

## 5 · (d) Alternative labels for workstream 2, and integration genes by zonation class

In [ ]:
pt_dist = pt[[c for c in pt.columns if c.startswith('glomerulus_distance')]].loc[labels_table.structure_id].reset_index(drop=True)
labels_table = pd.concat([labels_table, pt_dist], axis=1)
labels_table['glomerulus_attached_2um'] = labels_table.glomerulus_distance_um_morphological <= ATTACH_UM
labels_table.to_csv(output / 'alternative_labels.csv', index=False)
print('Saved', output / 'alternative_labels.csv', labels_table.shape)

pass2 = [p for p in sorted((upstream13 / 'stage_cache').glob('pass2_nephron_embedding__*.h5ad.key.json'))
         if "'logic': '13-nephron-pass2-v1'" in json.loads(p.read_text())['params']]
var = ad.read_h5ad(str(pass2[0])[:-len('.key.json')], backed='r').var
hvg752 = set(var.index[var.highly_variable_for_harmony.to_numpy(bool)])
pca611 = set(var.index[(var.highly_variable_for_harmony & var.highly_variable).to_numpy(bool)])
classes = pd.read_csv(results_root / 'pt_zonation_classes' / 'tables' / 'gene_zonation_classes.csv', index_col='gene')
class_rows = []
for column in ('class', 'S2-S1', 'S3c-early'):
    for label, genes in (('611 PCA genes', pca611), ('752 Harmony HVGs', hvg752)):
        counts = classes.loc[classes.index.intersection(sorted(genes)), column].value_counts()
        totals = classes[column].value_counts()
        for cls, n in totals.items():
            class_rows.append({'contrast': column, 'gene set': label, 'class': cls, 'genes in set': int(counts.get(cls, 0)),
                               'class size': int(n), 'share of class': float(counts.get(cls, 0) / n)})
CLASSES = pd.DataFrame(class_rows)
CLASSES.to_csv(output / 'integration_genes_by_class.csv', index=False)
display(CLASSES[CLASSES.contrast.eq('class')].pivot_table(index='class', columns='gene set',
        values=['genes in set', 'share of class']).round(3))

## 6 · D7 · Spillover from neighbouring cell types, and human polygons without bins

The spillover index is the fraction of a structure's counts from canonical non-PT markers. The
same index in non-PT structures shows that the markers mark those classes. Registration QC asks
whether polygons without bins lie outside the bin-covered region (capture area) or inside it.

In [ ]:
cohort_rebuilt = rebuild_pt_expression(
    cohort.assign(shared_pseudospace=.5)[['sample', 'comparison_species', 'shared_pseudospace']].astype(
        {'comparison_species': str}), source_paths, orthologs, target_sum=1e4)
C = sparse.csr_matrix(cohort_rebuilt.layers['counts'], dtype=float)
library = np.asarray(C.sum(axis=1)).ravel()
all_markers = [g for genes in MARKERS.values() for g in genes]
present = [g for g in all_markers if g in set(cohort_rebuilt.var_names[cohort_rebuilt.var.measured_in_both_inputs.to_numpy(bool)])]
nonpt = ~cohort.label.str.startswith('PT-').to_numpy()
idx = cohort_rebuilt.var_names.get_indexer(present)
detection_nonpt = np.asarray((C[nonpt][:, idx] > 0).mean(axis=0)).ravel()
used = [g for g, d in zip(present, detection_nonpt) if d >= .01]
print('Markers used:', used, '| dropped:', sorted(set(all_markers) - set(used)))
spill = {}
for category, genes in MARKERS.items():
    g = [x for x in genes if x in used]
    spill[category] = (np.asarray(C[:, cohort_rebuilt.var_names.get_indexer(g)].sum(axis=1)).ravel() / library) if g else np.zeros(len(library))
spill = pd.DataFrame(spill, index=cohort.index)
spill['total'] = spill.sum(axis=1)
frame = spill.join(cohort[['sample', 'comparison_species', 'label']])
SPILL = (frame.groupby(['comparison_species', 'label'])
         .agg(n=('total', 'size'), median=('total', 'median'), mean=('total', 'mean'),
              **{f'mean {c}': (c, 'mean') for c in MARKERS}).reset_index())
SPILL.to_csv(output / 'spillover_index.csv', index=False)
SPILL_SPECIMEN = frame[frame.label.str.startswith('PT-')].groupby(['sample', 'label']).total.agg(['median', 'mean']).reset_index()
SPILL_SPECIMEN.to_csv(output / 'spillover_index_by_specimen.csv', index=False)
display(SPILL[SPILL.label.str.startswith('PT-')].round(4))
display(SPILL[~SPILL.label.str.startswith('PT-')][['comparison_species', 'label', 'n', 'median', 'mean']].round(4))

qc_rows = []
for name, file in SEGMENTATION.items():
    features = json.loads((data_root / file).read_text())['features']
    native = ad.read_h5ad(source_paths[name], backed='r')
    obs = native.obs[['feature_index', 'n_spots']].copy()
    native.file.close()
    area_um2 = np.array([next(m['value'] for m in f['properties']['measurements'] if m['name'] == 'Area um^2') for f in features])
    polys = [shape(f['geometry']) for f in features]
    spots = np.zeros(len(features))
    spots[obs.feature_index.to_numpy(int)] = obs.n_spots.to_numpy(float)
    covered = spots > 0
    from shapely import MultiPoint
    hull = MultiPoint([p.centroid for p, c in zip(polys, covered) if c]).convex_hull
    inside = np.array([hull.contains(p.centroid) for p in polys])
    coverage = spots[covered] * MOUSE_BIN_UM ** 2 / area_um2[covered]
    quintile = pd.qcut(area_um2, 5, labels=False)
    in_cohort = cohort['sample'].eq(name).sum()
    qc_rows.append({'specimen': name, 'polygons': len(features), 'with zero bins': int((~covered).sum()),
                    'zero-bin fraction': float((~covered).mean()),
                    'zero-bin polygons inside bin-covered hull': int((~covered & inside).sum()),
                    'zero-bin fraction inside hull': float((~covered & inside).sum() / max(inside.sum(), 1)),
                    'in notebook 13 cohort': int(in_cohort),
                    'median coverage (bins x 4 um2 / area)': float(np.median(coverage)),
                    'coverage IQR': f'{np.percentile(coverage, 25):.2f}–{np.percentile(coverage, 75):.2f}',
                    **{f'zero-bin fraction, area quintile {q + 1}': float((~covered[quintile == q]).mean()) for q in range(5)}})
QC = pd.DataFrame(qc_rows)
QC.to_csv(output / 'registration_qc.csv', index=False)
display(QC.T)

## 7 · Figure and summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), layout='constrained')
morph = DISTANCE[DISTANCE.glomeruli.eq('morphological') & DISTANCE.segment.isin(['S1', 'S2', 'S3'])]
for i, (sp, color) in enumerate((('mouse', MOUSE), ('human', HUMAN))):
    block = morph[morph.species.eq(sp)]
    x = np.arange(3) + (i - .5) * .36
    axes[0].bar(x, block['median um'], width=.34, color=color, label=sp)
    axes[0].errorbar(x, block['median um'], yerr=[block['median um'] - block.q25, block.q75 - block['median um']],
                     fmt='none', ecolor=INK, lw=1)
axes[0].set(xticks=range(3), xticklabels=['S1', 'S2', 'S3'], ylabel='distance to nearest glomerulus (µm)',
            title='(a) Distance to glomerulus by label')
axes[0].legend(frameon=False)
att = ATTACHED[ATTACHED.glomeruli.eq('morphological') & ATTACHED['threshold um'].eq(ATTACH_UM) & ATTACHED.group.isin(SAMPLES)]
x = np.arange(2)
axes[1].bar(x - .18, att['S1 among all PT'], width=.34, color='#BBBBBB', label='all PT')
axes[1].bar(x + .18, att['S1 among attached'], width=.34, color=[MOUSE if g == 'mouse' else HUMAN for g in att.group],
            label='touching a glomerulus')
axes[1].set(xticks=x, xticklabels=list(att.group), ylabel='fraction labelled S1', ylim=(0, 1),
            title='(b) S1 among glomerulus-attached PT')
axes[1].legend(frameon=False)
prim = FRACTIONS[FRACTIONS.score.str.startswith('primary') & ~FRACTIONS.reference.str.startswith('male')]
labels_x = list(prim.specimen)
axes[2].bar(np.arange(len(prim)) - .18, prim['S1 fraction (current)'], width=.34, color='#BBBBBB', label='current labels')
axes[2].bar(np.arange(len(prim)) + .18, prim['S1 fraction (transfer)'], width=.34,
            color=[MOUSE if s == 'mouse' else HUMAN for s in prim.species], label='reference transfer')
axes[2].set(xticks=range(len(prim)), xticklabels=labels_x, ylabel='S1 fraction', ylim=(0, 1),
            title='(c) S1 fraction: current vs transferred')
axes[2].legend(frameon=False)
fig.savefig(figures / 'label_validation.pdf', bbox_inches='tight')
fig.savefig(figures / 'label_validation.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)

summary = {'logic_version': NOTEBOOK_LOGIC_VERSION, 'join': json.loads(JOIN.to_json(orient='records')),
           'glomeruli': json.loads(GLOM.to_json(orient='records')),
           'distance': json.loads(DISTANCE.to_json(orient='records')),
           'attached': json.loads(ATTACHED.to_json(orient='records')),
           'transfer': json.loads(TRANSFER.to_json(orient='records')),
           'fractions': json.loads(FRACTIONS.to_json(orient='records')),
           'pre-specified reading supports human labels': SUPPORT,
           'integration genes by class': json.loads(CLASSES.to_json(orient='records')),
           'spillover': json.loads(SPILL.to_json(orient='records')),
           'registration_qc': json.loads(QC.to_json(orient='records'))}
(output / 'summary.json').write_text(json.dumps(summary, indent=1, default=float))